# Stochastic Processes and Spike Trains

### NEUBEH/PBIO 545 — Quantitative Methods in Neuroscience

*Adapted from* [`matlab/stochasticProcessesTutorial.m`](../matlab/stochasticProcessesTutorial.m)
by Michael N. Shadlen and Greg Horwitz.

This tutorial is an introduction to **stochastic processes** and their application to spike
trains. It begins with *point processes* — lists of events that are indistinguishable from
one another, like spikes, radioactive decays, or grains of silver on film — and with the
special subclass called **renewal processes**, in which every event starts the process over
again.

We build up a statistical vocabulary for "disorderliness" along the way: the interval
distribution, the coefficient of variation, the count distribution and its Fano factor, the
**hazard function**, and **entropy**. These four descriptions do not always agree about
which process is the most random, and understanding *why* they disagree is the real payoff
of Part II. The last part changes topic: from processes made of identical events to
processes that move between meaningful **states** — Markov chains — ending with a random
walk toward absorbing barriers (gambler's ruin), the discrete cousin of diffusion.

| Part | Topic |
|---|---|
| I | Descriptive statistics of point and renewal processes |
| I — Example 1 | A renewal process that looks like a spike train (lognormal intervals) |
| I — Example 2 | The Poisson process: exponential intervals, Poisson counts, the hazard function |
| I — Example 3 | Gamma-distributed intervals: integrate-and-fire with $n$ steps to threshold |
| II | Entropy as a third measure of disorderliness |
| III | Markov chains: transition matrices, eigensystems, and gambler's ruin |

Run it cell by cell (**Shift+Enter**). The narrative and the homework questions are part of
the tutorial, not decoration.

**Prerequisites:** the basics of probability — random variable, expectation, variance,
probability density function, cumulative distribution function. If any of those words are
shaky, read the Berg chapter first: the one thing you must be able to do is compute the
expectation of $g(x)$ from the density $f(x)$.

**Prerequisite for Part III:** the linear algebra tutorial — eigenvectors and eigenvalues.

---
## Setup

Two conventions used throughout.

**Random numbers.** We fix a seed with `np.random.default_rng(...)` so every run of this
notebook reproduces the same figures. NumPy's PCG64 generator is *not* MATLAB's Mersenne
Twister, and the two are not seeded compatibly, so **the individual numbers printed here will
not match the numbers in the MATLAB tutorial** — only the statistics will. That is exactly
the point of a stochastic process: the sample is arbitrary, the distribution is not.

**Rasters.** The MATLAB tutorial depends on two plotting helpers, `plot1ras.m` (Shadlen) and
`plot2ras.m` (Horwitz). Both are reimplemented below in NumPy/matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.stats import lognorm, expon, gamma, poisson, binom, norm
from scipy.special import xlogy
import scipy.linalg

rng = np.random.default_rng(545)    # fixed seed, so every run reproduces the figures

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.titlesize": 11, "font.size": 9,
})

BLUE, RED, GREEN, PURPLE, GREY = "#3366d9", "#d94d3f", "#33914f", "#6a4fa8", "#808080"

# np.trapz was renamed np.trapezoid in NumPy 2.0; accept either.
trapezoid = getattr(np, "trapezoid", None) or np.trapz


def plot1ras(ax, times, trialnum=1, ticksize=0.85, color="k", lw=0.7):
    '''Python port of plot1ras.m (Shadlen & Meister, 1998).

    Draw one raster row: a vertical tick at each time in `times`, centred
    vertically on `trialnum`. NaNs (used to mark spikes past the end of the
    trial) are dropped, exactly as the MATLAB `finite()` call does.
    '''
    t = np.asarray(times, dtype=float).ravel()
    t = t[np.isfinite(t)]
    segs = [[(ti, trialnum - ticksize / 2), (ti, trialnum + ticksize / 2)] for ti in t]
    ax.add_collection(LineCollection(segs, colors=color, linewidths=lw))
    return t.size


def plot_raster(ax, arrivals, tmax=None, ticksize=0.85, color="k", lw=0.7):
    '''Raster for a whole (n_events, n_trials) array of arrival times.

    One column of `arrivals` = one trial = one row of the raster. Building a
    single LineCollection is far faster than looping plot1ras over 100 trials,
    but the geometry is identical.
    '''
    A = np.asarray(arrivals, dtype=float)
    if A.ndim == 1:
        A = A[:, None]
    segs = []
    for j in range(A.shape[1]):
        t = A[:, j]
        t = t[np.isfinite(t)]
        if tmax is not None:
            t = t[t <= tmax]
        y = j + 1                       # trials numbered from 1, as in MATLAB
        segs.extend([[(ti, y - ticksize / 2), (ti, y + ticksize / 2)] for ti in t])
    ax.add_collection(LineCollection(segs, colors=color, linewidths=lw))
    ax.set_ylim(0.5, A.shape[1] + 0.5)
    if tmax is not None:
        ax.set_xlim(0, tmax)
    ax.set_yticks([1] + list(range(10, A.shape[1] + 1, 10)))
    ax.grid(False)
    return ax


def density_hist(ax, data, edges, color=BLUE, label=None):
    '''Histogram normalized to a PROBABILITY DENSITY, drawn as bars.

    MATLAB's `hist` returns counts at bin CENTRES and the tutorial divides by
    sum(n) (and sometimes, incorrectly, by an unrelated bin width). NumPy's
    `np.histogram` takes bin EDGES and, with density=True, divides by
    count * binwidth so the bars integrate to 1 and can be compared directly
    with a pdf. That is what we do everywhere below.
    '''
    n, e = np.histogram(np.asarray(data).ravel(), bins=edges, density=True)
    ctr = 0.5 * (e[:-1] + e[1:])
    ax.bar(ctr, n, width=np.diff(e), color=color, edgecolor="none", label=label)
    return n, ctr

### MATLAB → Python translation notes

Most of the bugs you will hit porting this material come from a handful of calls whose
arguments *look* interchangeable and are not.

| MATLAB | Python | Watch out |
|---|---|---|
| `lognrnd(u,s)` | `rng.lognormal(u, s)` | Same convention: `u`,`s` are the mean/SD of the **underlying normal**. But `scipy.stats.lognorm` wants `lognorm(s, scale=exp(u))`. |
| `exprnd(mu)` | `rng.exponential(mu)` | Both take the **mean**. `scipy.stats.expon` wants `scale=mu` — never `expon(mu)`, which sets the *location*. |
| `gamrnd(a,b)` | `rng.gamma(a, b)` | Both are (shape, **scale**). Many textbooks and some libraries use a *rate* $\beta = 1/b$ instead. |
| `poissrnd(lam)` | `rng.poisson(lam)` | Same. |
| `hist(x, ctrs)` | `np.histogram(x, edges)` | MATLAB takes bin **centres**, NumPy takes bin **edges**, and MATLAB's last bin absorbs everything above it while NumPy's does not. |
| `P(2,:)` | `P[1, :]` | 1-based vs 0-based. In Part III this means "state 2" is row index 1. |
| `A^n` | `np.linalg.matrix_power(A, n)` | **The silent-wrong-answer trap.** In MATLAB `A^n` is a *matrix* power and `A.^n` is elementwise. In NumPy it is the other way round: `A**n` is **elementwise** and `A @ A` / `matrix_power` are the matrix operation. Nothing errors — you just get a different, plausible-looking matrix. |
| `A*B` | `A @ B` | `*` is elementwise in NumPy. |
| `nansum` | `xlogy` / `np.nansum` | For entropy, `scipy.special.xlogy(p, p)` gives the correct $0\log 0 = 0$ without a warning. |
| `sqrtm`, `expm` | `scipy.linalg.sqrtm`, `expm` | Matrix square root / exponential — again not elementwise `np.sqrt`, `np.exp`. |

---
## Part I. Descriptive statistics

**What is a point process?** It is a list of events that are indistinguishable from one
another — spikes, radioactive decays, the onset of an alarm, a grain of silver on a piece of
film. Each *event* has a time (or a location), so the process is fully described by a list of
times, often called **arrival times**.

In important special cases we choose instead to characterize the **intervals** between those
times, and/or the **number of events** in some epoch. Those two descriptions are especially
useful when the random machinery generating the events does not change with time.

**What is a renewal process?** A point process in which every event starts the process all
over again. Suppose the events are failures of your hard drive; every time it fails you
replace it with a new one. Whatever process describes the time to failure just plays itself
out again from scratch. So: *a renewal process is a point process whose intervals are drawn
independently from a common distribution* — the intervals are **iid** (independent and
identically distributed).

Note what iid does *not* mean. It does not mean the interval distribution has to be simple.
Your hard drive might have one failure mode about an hour into use and another spread over
months to years; the interval distribution would then be bimodal and ugly. As long as a
*new* drive's time to failure is described by that same distribution every time, you have a
renewal process.

---
### Example 1. A renewal process that looks like a spike train

We draw interevent intervals from a **lognormal** distribution. If $X$ is lognormal then
$\log X$ is normal — hence the name. Its density is

$$f(x) = \frac{1}{x\sigma\sqrt{2\pi}}\,
        \exp\!\left[-\frac{(\ln x - \mu)^2}{2\sigma^2}\right], \qquad x > 0$$

with $\mu = 2.5$, $\sigma = 1$ here. (The choice $\mu = 2.5$ is not arbitrary; it puts the
mean interval near 20 ms, i.e. a rate near 50 spikes/s, which is what we will use for every
process in this tutorial so the comparisons are fair.)

The density is non-zero only for **positive** durations, which it had better be — the time
between two events cannot be negative. Plotting it against $\log x$ makes the underlying
Gaussian visible.

In [ ]:
dx = 0.001
x  = np.arange(dx, 200 + dx, dx)
u, s = 2.5, 1.0                       # mu and sigma of the UNDERLYING normal

# scipy parameterization: lognorm(s, scale=exp(u))  <->  MATLAB lognpdf(x, u, s)
y = lognorm.pdf(x, s, scale=np.exp(u))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(x, y, color=BLUE)
axes[0].set(xlabel="Interval duration (ms)", ylabel="Probability density",
            xlim=(0, 200), title="The lognormal distribution")
axes[1].plot(x, y, color=BLUE)
axes[1].set_xscale("log")
axes[1].set(xlabel="Interval duration (ms, log scale)", ylabel="Probability density",
            title="Same density on a log abscissa: a Gaussian")
fig.tight_layout()

m_theory = np.exp(u + s**2 / 2)                                  # lognstat(u,s)
v_theory = (np.exp(s**2) - 1) * np.exp(2*u + s**2)
print(f"theoretical mean interval = {m_theory:.3f} ms  ->  rate = {1000/m_theory:.2f} spikes/s")
print(f"theoretical SD            = {np.sqrt(v_theory):.3f} ms")
print(f"theoretical CV            = {np.sqrt(v_theory)/m_theory:.4f}")

Now imagine a neuron whose interspike intervals (ISIs) are lognormal. We simulate spike
trains by drawing a pile of ISIs at random and calling each event a spike, repeating 100
times for 100 trials.

In [ ]:
ntrials     = 100
n_intervals = 300
intervals   = rng.lognormal(u, s, size=(n_intervals, ntrials))   # (events, trials)

edges = np.arange(0, 501, 1.0)          # 1-ms bins, EDGES (MATLAB used centres 0:500)

fig, ax = plt.subplots(figsize=(8, 3.6))
density_hist(ax, intervals, edges, label="simulated ISIs")
ax.plot(x, y, color=RED, lw=2, label="lognormal pdf")
ax.set(xlabel="Interval (ms)", ylabel="Probability density", xlim=(0, 200),
       title=f"ISI histogram, {intervals.size:,} intervals")
ax.legend()
fig.tight_layout()

print(f"fraction of intervals longer than 200 ms: {(intervals > 200).mean():.4f}")
print(f"longest interval drawn:                   {intervals.max():.1f} ms")

As expected the histogram looks like the density it was drawn from. Note the **long tail**:
a small but non-negligible fraction of intervals are many times the mean. That tail is a
characteristic we will come back to — it is what makes this process *more* disorderly than a
Poisson process by the CV measure.

Now use a cumulative sum to turn intervals into arrival times, and draw the raster. Spikes
beyond `tmax` are set to `NaN` and dropped by the raster routine.

In [ ]:
arrivals = np.cumsum(intervals, axis=0)
tmax     = 1000
arrivals_logn = np.where(arrivals > tmax, np.nan, arrivals)

n_show = 50          # 100 trials would be too dense to see individual rows
fig, ax = plt.subplots(figsize=(9.5, 6))
plot_raster(ax, arrivals_logn[:, :n_show], tmax=tmax, ticksize=0.7)
ax.set(xlabel="Time (ms)", ylabel="Trial number",
       title=f"Renewal process with lognormal intervals — first {n_show} trials")
fig.tight_layout()

print(f"mean spikes per trial in the first {tmax} ms: "
      f"{np.isfinite(arrivals_logn).sum(axis=0).mean():.1f}")

The arrivals look pretty irregular. Notice the clumps: bursts of closely spaced ticks
separated by long empty gaps. That is the long tail of the lognormal showing up in the time
domain.

*(The MATLAB tutorial plays the spike trains through the computer speaker at this point —
what an electrophysiologist hears over the audio monitor. The `sound` call was already
broken on the author's machine; we skip it. If you want to hear it, bin the arrival times at
1 ms and hand the vector to `IPython.display.Audio(rate=1000)`.)*

#### Interval statistics

The ratio of standard deviation to mean is the **coefficient of variation**,

$$\mathrm{CV} = \frac{\sigma}{\mu}.$$

In the right context it is the reciprocal of the signal-to-noise ratio.

In [ ]:
mean_int = intervals.mean()
var_int  = intervals.var(ddof=1)
CV_int   = np.sqrt(var_int) / mean_int

print(f"empirical mean interval = {mean_int:8.3f} ms   (theory {m_theory:.3f})")
print(f"implied rate            = {1000/mean_int:8.3f} spikes/s")
print(f"empirical variance      = {var_int:8.3f} ms^2  (theory {v_theory:.3f})")
print(f"empirical CV            = {CV_int:8.4f}        (theory {np.sqrt(v_theory)/m_theory:.4f})")

Keep that CV in the back of your mind. In a moment we compare it with the CV of a very
special interval distribution — the exponential.

> **Question.** The CV is the reciprocal of a signal-to-noise ratio. Can you think of
> situations in neuroscience where SNR really is $\mu/\sigma$ of a single measured quantity,
> and situations where that identification would be misleading?

#### Count statistics

Neurophysiologists usually work with **counts**, not intervals. The expected count in an
epoch is just the epoch duration divided by the expected interval — but of course we do not
get the same count every time. The interest is in the *variability* of the count.

Each trial here contains 300 spikes and therefore ends at a different time, so we count
within the first 500 ms, short enough that even the shortest trial covers it.

In [ ]:
epochDur = 500
assert arrivals[-1, :].min() > epochDur, "Not enough spikes!"
counts = (arrivals < epochDur).sum(axis=0)        # counts per trial, lognormal neuron

mean_counts = counts.mean()
var_counts  = counts.var(ddof=1)

fig, ax = plt.subplots(figsize=(7, 3.4))
cedges = np.arange(counts.min() - 1, counts.max() + 3, 2.0)
ax.hist(counts, bins=cedges, color=BLUE, edgecolor="white")
ax.set(xlabel="Number of spikes in epoch", ylabel="Number of trials",
       title=f"Spike counts in {epochDur} ms — lognormal neuron")
fig.tight_layout()

print(f"mean count       = {mean_counts:8.3f}")
print(f"implied rate     = {mean_counts/(epochDur/1000):8.3f} spikes/s")
print(f"count variance   = {var_counts:8.3f}")

For **any** renewal process there is a remarkable asymptotic relationship between the
interval statistics and the count statistics. Writing $T$ for the counting window and
$\mu_I, \sigma_I^2$ for the mean and variance of the interval distribution,

$$\mathbb{E}[N] \;\to\; \frac{T}{\mu_I}, \qquad
  \mathrm{Var}[N] \;\to\; \frac{T\,\sigma_I^2}{\mu_I^{3}}.$$

Equivalently, the **Fano factor** of the counts converges to the **squared CV** of the
intervals:

$$F \;=\; \frac{\mathrm{Var}[N]}{\mathbb{E}[N]} \;\longrightarrow\; \mathrm{CV}^2 .$$

The mean relation converges quickly. The variance relation converges *much* more slowly, and
with $T = 500$ ms — only about 25 intervals — you should expect a visible discrepancy.

In [ ]:
print("mean of counts")
print(f"   asymptotic prediction  T/mu_I        = {epochDur/mean_int:8.3f}")
print(f"   measured               mean(counts)  = {mean_counts:8.3f}")
print()
print("variance of counts")
print(f"   asymptotic prediction  T*var_I/mu_I^3 = {epochDur*var_int/mean_int**3:8.3f}")
print(f"   measured               var(counts)    = {var_counts:8.3f}")
print()
print("CV^2 of intervals vs Fano factor of counts")
print(f"   CV_int^2                  = {CV_int**2:8.4f}")
print(f"   var(counts)/mean(counts)  = {var_counts/mean_counts:8.4f}")
print(f"   ratio (should -> 1)       = {(var_counts/mean_counts)/CV_int**2:8.4f}")

The two numbers in each pair are in the same ballpark but not equal, and the count variance
is the worse of the two — as advertised. The asymptotic identity is a statement about
$T \to \infty$; with ~25 intervals per epoch, and with a heavy-tailed interval distribution
feeding it, convergence is slow and the sampling error on a variance estimated from 100
trials is itself large. **Do not read the numbers above as a demonstration that
$F = \mathrm{CV}^2$ exactly; read them as a demonstration that the two quantities track each
other.**

One more general fact before we leave renewal processes: as the counting window grows long
compared with the mean interval, the count distribution becomes more and more **Gaussian**,
by the central limit theorem — a sum of many iid intervals is what sets the count. Let us
check that directly.

In [ ]:
# Long counting windows: does the count distribution become Gaussian?
from scipy.stats import skew as sample_skew

n_long   = 8000
windows  = [200, 1000, 5000, 20000]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
print(f"{'T (ms)':>8} {'mean':>9} {'var':>10} {'Fano':>7} {'skew':>8}")
for ax, T in zip(axes, windows):
    n_need = int(3 * T / m_theory + 80)
    arr = np.cumsum(rng.lognormal(u, s, size=(n_need, n_long)), axis=0)
    c   = (arr < T).sum(axis=0)
    assert np.isfinite(arr[-1]).all() and arr[-1].min() > T, "trials too short"
    ax.hist(c, bins=np.arange(c.min() - 0.5, c.max() + 1.5), density=True,
            color=BLUE, edgecolor="none")
    grid = np.linspace(c.min(), c.max(), 300)
    ax.plot(grid, norm.pdf(grid, c.mean(), c.std(ddof=1)), color=RED, lw=2)
    ax.set(title=f"T = {T} ms  (mean {c.mean():.0f})",
           xlabel="Count", ylabel="Density" if T == windows[0] else "")
    print(f"{T:>8d} {c.mean():>9.2f} {c.var(ddof=1):>10.2f} "
          f"{c.var(ddof=1)/c.mean():>7.3f} {float(sample_skew(c)):>+8.3f}")
fig.suptitle("Counts from a renewal process become Gaussian as the window grows", y=1.04)
fig.tight_layout()
print(f"\nCV^2 of the lognormal intervals (theory) = {v_theory/m_theory**2:.4f}"
      "   <- the Fano factors above are converging to this")

The **Fano factor** climbs monotonically toward $\mathrm{CV}^2 \approx 1.72$ as $T$ grows:
the asymptotic identity really does hold, it just needs a long window — at $T = 500$ ms,
where we did the count statistics above, it is still well short.

The approach to Gaussianity is worth reading carefully, because the numbers do **not** do the
simple thing. The sample skew is not monotone in $T$: it is near zero at $T = 200$ ms,
reaches about $-0.19$ near $T = 1000$ ms, and only then decays back toward zero. The dip is
the lognormal's heavy tail at work — a single very long interval costs you several counts, so
the count distribution grows a left tail — and the reason it is *invisible* at $T = 200$ ms is
that the mean count there is only about 10 and the count cannot go below zero, so the floor
truncates the very tail that would have produced the skew. Beyond the dip the central limit
theorem takes over and the skew decays roughly as $1/\sqrt{T}$. The lesson: "converges to a
Gaussian" is an asymptotic statement, and the route there can be non-monotone.

---
### Example 2. A Poisson process

A **Poisson process** is a renewal process whose interevent times are **exponentially**
distributed:

$$f(t) = \frac{1}{\mu}e^{-t/\mu} = \lambda e^{-\lambda t}, \qquad t \ge 0$$

where $\lambda = 1/\mu$ is the rate. Its mean and standard deviation are both $\mu$, so

$$\mathrm{CV} = 1 \quad \text{exactly, for every rate.}$$

To simulate a Poisson neuron we draw exponential intervals and place spikes at the running
sum. We use 50 spikes/s, so the mean interval is 20 ms — the same rate as the lognormal
neuron above, which is what makes the two comparable.

In [ ]:
ntrials    = 100
spike_rate = 50                      # spikes/s
meanint    = 1000 / spike_rate       # mean ISI (ms)

# MATLAB exprnd(mu) takes a MEAN; rng.exponential(scale) takes the same thing.
intervals_p = rng.exponential(meanint, size=(100, ntrials))

fig, ax = plt.subplots(figsize=(8, 3.6))
density_hist(ax, intervals_p, edges, label="simulated ISIs")
tt = np.linspace(0, 200, 400)
ax.plot(tt, expon.pdf(tt, scale=meanint), color=RED, lw=2, label="exponential pdf")
ax.set(xlabel="Interval duration (ms)", ylabel="Probability density", xlim=(0, 200),
       title="ISI histogram, Poisson neuron")
ax.legend()
fig.tight_layout()

print(f"empirical mean ISI = {intervals_p.mean():.3f} ms   (theory {meanint})")
print(f"empirical CV       = {intervals_p.std(ddof=1)/intervals_p.mean():.4f}   (theory 1)")

The empirical CV is close to 1 but not equal to it; with 10,000 intervals the standard error
on the CV is about $1/\sqrt{2n} \approx 0.7\%$, so a value a percent or so off 1 is exactly
what you should expect. **The CV of a Poisson process is 1 as a statement about the
distribution, not about any finite sample.**

*(A note on the original: the MATLAB code normalized this histogram with
`n = n/(sum(n)*.1)` — a bin width of 0.1 copied from `plot2ras.m`, where it was correct —
while the bins here are 1 ms wide. The plotted "probability" was therefore ten times too
large and did not match the pdf. We normalize with the actual bin width.)*

In [ ]:
arrivals_p = np.cumsum(intervals_p, axis=0)
tmax = 1000
arrivals_pois = np.where(arrivals_p > tmax, np.nan, arrivals_p)

fig, ax = plt.subplots(figsize=(9.5, 6))
plot_raster(ax, arrivals_pois[:, :n_show], tmax=tmax, ticksize=0.7)
ax.set(xlabel="Time (ms)", ylabel="Trial number",
       title=f"Poisson process, 50 spikes/s — first {n_show} trials "
             "(same axes as the lognormal raster)")
fig.tight_layout()

The Poisson rasters look qualitatively much like the lognormal ones, and over a speaker they
sound about the same. Our eyes and ears are simply not good at telling these two random
processes apart. The *statistics*, however, are quite different — most visibly in the counts.

In [ ]:
epochDur = 500
assert arrivals_p[-1, :].min() > epochDur, "Not enough spikes!"
poisscounts = (arrivals_p < epochDur).sum(axis=0)

bin0, binLast, binwidth = min(counts.min(), poisscounts.min()), \
                          max(counts.max(), poisscounts.max()), 2
cedges = np.arange(bin0 - binwidth/2, binLast + 1.5*binwidth, binwidth)
ctr    = 0.5 * (cedges[:-1] + cedges[1:])

fig, axes = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True, sharey=True)
for ax, c, ttl, col in ((axes[0], poisscounts, "Poisson neuron", BLUE),
                        (axes[1], counts, "non-Poisson (lognormal) neuron", PURPLE)):
    n, _ = np.histogram(c, bins=cedges)
    ax.bar(ctr, n / n.sum(), width=binwidth, color=col, edgecolor="white")
    ax.set(ylabel="Proportion of trials", title=ttl)
# analytic Poisson pmf, summed over each 2-wide bin
axes[0].plot(ctr, binwidth * poisson.pmf(ctr, spike_rate * epochDur/1000), "*",
             color=RED, ms=8, label="Poisson pmf (theory)")
axes[0].legend()
axes[1].set_xlabel("Number of spikes in epoch")
fig.tight_layout()

for name, c in (("Poisson  ", poisscounts), ("lognormal", counts)):
    print(f"{name}: mean={c.mean():7.2f}  var={c.var(ddof=1):8.2f}  "
          f"Fano={c.var(ddof=1)/c.mean():6.3f}  SD={c.std(ddof=1):6.2f}")

Both panels share axis limits, so the widths are directly comparable. The Poisson neuron's
count, although random, is **more consistent** trial to trial than the lognormal neuron's.
That is a direct consequence of the interval CVs: $\mathrm{CV}=1$ for the exponential versus
$\mathrm{CV}\approx 1.31$ for this lognormal, and $F \approx \mathrm{CV}^2$.

Look at the Poisson Fano factor printed above. It is near 1 — as theory demands for a Poisson
process, where $\mathrm{Var}[N] = \mathbb{E}[N]$ exactly — but with only 100 trials the
sampling error on a variance is roughly $\sqrt{2/(n-1)} \approx 14\%$, so a value between
about 0.85 and 1.15 is unremarkable. Again: **1 is the property of the process, not of the
sample.**

The red stars are the analytic Poisson probability

$$P\{X(t) = k\} = \frac{(\lambda t)^k e^{-\lambda t}}{k!}$$

evaluated at each bin centre and multiplied by the bin width. It tracks the Poisson neuron's
histogram well and does not describe the lognormal neuron's counts at all. (The count
distribution for a general renewal process has no nice closed form.)

#### Where the Poisson distribution comes from

The Poisson distribution is the limit of a sequence of independent binary trials in which
only the overall rate is known. Suppose the rate is 10 events per second and I ask for the
probability of an event in any given millisecond. The answer had better be 1 in 100. If
every millisecond is independent of every other, the count in one second is **binomial**.

In [ ]:
N = 1000                    # millisecond bins in one second
p = 10 / N                  # probability of an event in one bin
q = 1 - p
k = np.arange(0, 31)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.stem(k, binom.pmf(k, N, p), linefmt=BLUE, markerfmt="o", basefmt=" ",
        label=f"Binomial(N={N}, p={p})")
ax.plot(k + 0.18, poisson.pmf(k, N * p), "^", color=RED, ms=5,
        label=f"Poisson(lambda={N*p:.0f})")
ax.set(xlabel="Count", ylabel="Probability",
       title="The binomial converges to the Poisson as N grows and p shrinks")
ax.legend()
fig.tight_layout()

print(f"expected number of events  N*p     = {N*p:.3f}")
print(f"binomial mean, variance            = {N*p:.4f}, {N*p*q:.4f}")
print(f"Poisson  mean, variance            = {N*p:.4f}, {N*p:.4f}")
print(f"max |binomial - Poisson| over k    = {np.abs(binom.pmf(k,N,p)-poisson.pmf(k,N*p)).max():.2e}")

> **Question.** Why plot this with `stem` rather than `plot` or `bar`? (Hint: what kind of
> quantity is $P\{X=k\}$, and for which values of $k$ is it defined?)

Notice that the binomial variance $Npq$ is very slightly *smaller* than the Poisson variance
$Np$, by exactly the factor $q$. As $p \to 0$ with $Np$ fixed — the "bin size goes to zero,
$N$ goes to infinity" limit — that difference vanishes, the binomial becomes Poisson, and
the waiting time between events becomes exponential.

#### Memorylessness and the hazard function

The second remarkable property of the Poisson process is about its intervals. Consider a
spike at $t = 57$ ms with a rate of 50 events/s. The probability that the next event happens
in the next moment is the same at 58 ms as at 90 ms, *given that we are still waiting*. We
learn **nothing** about the imminence of the next event from the mere fact that time has
passed. The process is **memoryless**.

To make that precise, define the **hazard function** — the probability that an event occurs
at $t$ *given that it has not occurred yet*:

$$h(t) \;=\; p(t \mid \text{"not yet"}) \;=\; \frac{p(t, \text{"not yet"})}{p(\text{"not yet"})}
        \;=\; \frac{f(t)}{1 - F(t)}$$

The numerator is the interval density itself: the probability of an interval of exactly
length $t$ *is* the probability that the event happens at $t$ and has not happened before.
The denominator is the survivor function, one minus the cumulative distribution.

In [ ]:
t = np.arange(0, 200.1, 0.1)

# h(t) = f(t) / [1 - F(t)]. Use the SURVIVAL function sf = 1 - cdf rather than
# writing 1 - cdf literally: at large t the cdf is 1 to machine precision and the
# subtraction cancels catastrophically, giving 0/0 or divide-by-zero garbage.
hazard_exp  = expon.pdf(t, scale=meanint) / expon.sf(t, scale=meanint)
hazard_logn = lognorm.pdf(t, s, scale=np.exp(u)) / lognorm.sf(t, s, scale=np.exp(u))

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(t, hazard_exp, color=BLUE, lw=2, label="exponential (Poisson process)")
ax.plot(t, hazard_logn, color=RED, lw=2, label="lognormal")
ax.axhline(1/meanint, color=GREY, ls=":", lw=1)
ax.set(xlabel="Time since previous spike (ms)",
       ylabel="P(spike now | no spike yet)  [per ms]",
       ylim=(0, 0.1), xlim=(0, 200), title="Hazard functions")
ax.legend()
fig.tight_layout()

print(f"exponential hazard: min={np.nanmin(hazard_exp):.5f}  max={np.nanmax(hazard_exp):.5f}  "
      f"1/mu={1/meanint:.5f}   (flat, as theory requires)")
i = np.nanargmax(hazard_logn)
print(f"lognormal hazard peaks at t = {t[i]:.1f} ms, h = {hazard_logn[i]:.4f} per ms")
print(f"lognormal hazard at t = 200 ms: {hazard_logn[-1]:.5f} per ms  (still falling)")

The exponential hazard is **totally flat** at exactly $1/\mu = 0.05$ per ms. The probability
of firing in the next small increment $dt$ is $h\,dt$ regardless of how long we have waited.
That flatness is the unique signature of the exponential distribution: no other interval
distribution has a constant hazard.

The lognormal hazard tells a different story. It rises to a peak at about 7.5 ms — there is a
*favoured* time for the next spike — and then falls monotonically: once that moment has
passed, the longer you have waited, the *less* likely a spike becomes in the next moment.
That is the long tail again, seen from the conditional point of view. (The MATLAB original
puts the peak at "~12 msec"; the computed peak for these parameters is 7.5 ms. The
qualitative claim — a favoured interval followed by monotone decline — is unaffected.)

So we now have two measures that disagree. By the **CV**, the lognormal process is the more
disorderly (CV 1.31 vs 1). By the **hazard**, the Poisson process is the more unpredictable —
nothing at all can be learned by waiting. Hold that tension; Part II adds a third measure.

> ### Homework question 1
> A sprinter is waiting for the 'go' signal to start a race. The 'go' is the second of two
> pips (clicks). The interval between pips is 1 s on average, but random: any time from 0.5
> to 1.5 s is equally probable (a uniform distribution).
>
> **(a)** Sketch the shape of the hazard function. You do not have to derive it — but it is
> easy if you try. What happens as $t \to 1.5$ s, and why?
>
> **(b)** If the sprinter's nervous system has the knowledge you have, could it affect her
> start time? How would we tell, experimentally?
>
> **(c)** Redraw the sketch for a pip interval drawn from an *exponential* distribution with
> the same mean. What would the sprinter's reaction time look like then?

> ### Homework question 2
> What is the general relationship between a probability density $f(t)$ and its associated
> cumulative distribution $F(t)$? Write it both ways (as a derivative and as an integral),
> and then use it to show that $h(t) = f(t)/[1-F(t)]$ can also be written
> $h(t) = -\frac{d}{dt}\ln[1 - F(t)]$.

---
### Example 3. Gamma-distributed interevent intervals

Now we simulate a renewal process by a completely different route. Imagine a neuron that
integrates excitatory postsynaptic potentials arriving as a Poisson process. Once $n$ EPSPs
have accumulated, it fires a spike and resets. This is a renewal process — intervals are iid
— and we did not stipulate the interval distribution at all. It turns out to be the
**gamma** distribution:

$$f(t) = \frac{t^{n-1}e^{-t/b}}{b^{n}\,\Gamma(n)}$$

with shape $n$ (the number of steps to threshold) and scale $b$ (the mean EPSP interval).
Its mean is $nb$, its variance $nb^2$, and therefore

$$\mathrm{CV} = \frac{\sqrt{n}\,b}{nb} = \frac{1}{\sqrt{n}}.$$

For $n = 1$ this is the exponential (CV = 1) and the process is Poisson. Larger $n$ means a
more regular spike train.

We hold the *output* rate fixed at 50 spikes/s by scaling the EPSP rate with $n$: each EPSP
interval has mean $\mu/n$, so $n$ of them still sum to $\mu = 20$ ms. Without that
correction the model neuron would simply fire more and more slowly as $n$ grew.

In [ ]:
def gamma_neuron(nstepstothresh, n_isi_per_trial=50, ntrials=20, n_extra=0, rng=rng):
    '''Simulate an integrate-to-threshold neuron: sum n exponential EPSP intervals.

    Returns (intervals, arrivals) with intervals shaped (n_isi_per_trial, ntrials).
    MATLAB: exprnd(meanint/n, n, N) then sum down the columns.
    '''
    N = n_isi_per_trial * ntrials + n_extra
    epsp = rng.exponential(meanint / nstepstothresh, size=(nstepstothresh, N))
    isi  = epsp.sum(axis=0)                      # one ISI per column
    isi_mat = isi[:n_isi_per_trial*ntrials].reshape(n_isi_per_trial, ntrials, order="F")
    return isi, np.cumsum(isi_mat, axis=0)


steps_list = [1, 5, 20]
ras_tmax   = 500
isi_edges  = np.arange(0, 60.5, 1.0)
tg         = np.linspace(0.01, 60, 400)

fig, axes = plt.subplots(3, 2, figsize=(11, 7.5),
                         gridspec_kw={"width_ratios": [1, 1.3]})
sim = {}
for r, nsteps in enumerate(steps_list):
    isi, arr = gamma_neuron(nsteps)
    sim[nsteps] = (isi, arr)
    axL, axR = axes[r]
    density_hist(axL, isi, isi_edges, color=BLUE)
    # MATLAB gampdf(t, a, b) == scipy gamma.pdf(t, a, scale=b); b is a SCALE, not a rate.
    axL.plot(tg, gamma.pdf(tg, nsteps, scale=meanint/nsteps), color=GREEN, lw=2)
    axL.set(xlim=(0, 60), ylim=(0, 0.18), ylabel="Probability density")
    axL.text(0.97, 0.9, f"{nsteps} step{'s' if nsteps > 1 else ''} to threshold\n"
                        f"CV = {isi.std(ddof=1)/isi.mean():.2f}",
             transform=axL.transAxes, ha="right", va="top")
    plot_raster(axR, arr, tmax=ras_tmax)
    axR.set(ylabel="Trial")
axes[2, 0].set_xlabel("Interval (ms)")
axes[2, 1].set_xlabel("Time (ms)")
axes[0, 0].set_title("Interspike interval distribution (green = gamma pdf)")
axes[0, 1].set_title("Spike rasters, 20 trials")
fig.tight_layout()

print(f"{'n':>4} {'mean ISI':>10} {'CV emp':>8} {'CV=1/sqrt(n)':>13} {'rate (sp/s)':>12}")
for nsteps in steps_list:
    isi = sim[nsteps][0]
    print(f"{nsteps:>4} {isi.mean():>10.3f} {isi.std(ddof=1)/isi.mean():>8.3f} "
          f"{1/np.sqrt(nsteps):>13.3f} {1000/isi.mean():>12.2f}")

Read the figure **down the columns**. As the neuron is required to integrate more EPSPs, the
ISI histogram gets tighter and the spike trains look progressively more regular — the ticks
in the bottom raster are nearly evenly spaced. The mean rate is the same in all three rows;
only the regularity changes.

With one step to threshold there are plenty of very short and very long intervals: it is
easy for two Poisson events to land nearly simultaneously. With five steps, very short
intervals nearly disappear — five Poisson events arriving almost together is possible in
principle, but exceedingly unlikely. The green curves confirm that the interval distribution
of the integrate-to-threshold model really is a gamma density, which means we could have
skipped the EPSP simulation entirely and drawn ISIs from `rng.gamma(n, meanint/n)` directly.

**A caution about the model.** The CV of the intervals falls as $1/\sqrt{n}$, and because
$F \approx \mathrm{CV}^2$, the spike-count variance falls with it. This is a real problem for
the simple integrate-and-fire account of cortical firing: it predicts responses far more
regular than what is actually recorded in vivo, where CVs near 1 are typical. The usual
resolution is that real neurons receive massive *inhibition* as well as excitation; a
balanced random walk toward threshold restores the irregularity.

Let us verify the "we could have drawn from a gamma directly" claim, and then look at the
densities and hazards analytically.

In [ ]:
# The from-scratch EPSP simulation vs. the theoretical gamma, and vs. direct gamma draws.
from scipy.stats import kstest

print(f"{'n':>4} {'KS stat':>9} {'p-value':>9} {'mean(EPSP sim)':>15} {'mean(gamma draw)':>17}"
      f" {'theory':>8}")
for nsteps in steps_list:
    isi = sim[nsteps][0]
    # MATLAB gamrnd(a, b) == rng.gamma(a, b): both are (shape, SCALE).
    direct = rng.gamma(nsteps, meanint/nsteps, size=isi.size)
    ks = kstest(isi, gamma(nsteps, scale=meanint/nsteps).cdf)
    print(f"{nsteps:>4} {ks.statistic:>9.4f} {ks.pvalue:>9.3f} "
          f"{isi.mean():>15.3f} {direct.mean():>17.3f} {meanint:>8.1f}")
print("\nKS test of the summed-EPSP intervals against the theoretical gamma CDF:")
print("large p-values mean we cannot reject the gamma, which is the point. The two")
print("simulation routes -- summing n exponentials, or one gamma draw -- agree.")

In [ ]:
t = np.arange(0, 200.1, 0.1)

fig, axes = plt.subplots(3, 2, figsize=(10, 7), sharex=True)
for r, nsteps in enumerate(steps_list):
    a, b = nsteps, meanint / nsteps
    pdf = gamma.pdf(t, a, scale=b)
    haz = pdf / gamma.sf(t, a, scale=b)           # sf, NOT 1-cdf: see the note above
    m, v = a * b, a * b**2                        # gamstat(a, b)
    axes[r, 0].plot(t, pdf, color=BLUE, lw=1.8)
    axes[r, 0].text(0.97, 0.88, f"n = {nsteps}\nCV = {np.sqrt(v)/m:.2f}",
                    transform=axes[r, 0].transAxes, ha="right", va="top")
    axes[r, 1].plot(t, haz, color=RED, lw=1.8)
    axes[r, 1].axhline(1/meanint, color=GREY, ls=":", lw=1)
    axes[r, 0].set(xlim=(0, 100), ylim=(0, 0.1), ylabel="Density")
    axes[r, 1].set(xlim=(0, 100), ylim=(0, 1.1), ylabel="Hazard (per ms)")
axes[0, 0].set_title("Interval distribution (pdf)")
axes[0, 1].set_title("Hazard rate")
axes[2, 0].set_xlabel("Interval (ms)")
axes[2, 1].set_xlabel("Time since last spike (ms)")
fig.tight_layout()

for nsteps in steps_list:
    a, b = nsteps, meanint/nsteps
    h = gamma.pdf(t, a, scale=b) / gamma.sf(t, a, scale=b)
    print(f"n={nsteps:>3}:  hazard at t=0 is {h[0]:.4f},  at t=100 ms it is {h[1000]:.4f},  "
          f"asymptote 1/b = {1/b:.4f} per ms")

The first-order gamma *is* the exponential, so its hazard is the flat line we already know.
For $n>1$ the hazard starts at **zero** — immediately after a spike the neuron has
accumulated no EPSPs, so it cannot fire — climbs, and then saturates at $1/b$, the EPSP
arrival rate. That asymptote makes mechanistic sense: if you have waited a very long time you
have certainly collected $n-1$ EPSPs already, so the only thing standing between you and a
spike is the next EPSP, which arrives at rate $1/b$. For $n=20$ that asymptote is 1 per ms —
twenty times the flat Poisson hazard — which is the analytic face of "these spike trains look
almost like a metronome".

Note the axis limits: all three pdf panels share $y \in [0, 0.1]$ and all three hazard panels
share $y \in [0, 1.1]$, so the rows are genuinely comparable. (The MATLAB original caps the
hazard axis at 0.8, which clips the $n=20$ curve, and computes the hazard as
`gampdf ./ (1 - gamcdf)`; past about 75 ms the cdf is 1 in double precision and that
expression turns into numerical noise. Using the survival function fixes both.)

---
## Part II. Entropy

We have been talking about disorderliness, and two measures have disagreed. By the **CV**,
the Poisson process is a benchmark at exactly 1: the lognormal was more disorderly (CV > 1),
the gammas less (CV < 1). By the **hazard function**, the Poisson process looked like the
most random of all — no interval is preferred and nothing is learned by waiting, a property
unique to the exponential.

Here is a third way to think about it: **entropy**, the average of $-\log_2 p$,

$$H = -\sum_i p_i \log_2 p_i .$$

The units are **bits**. Start with a fair coin.

In [ ]:
def entropy_bits(p):
    '''Shannon entropy in bits, with the convention 0*log(0) = 0.

    MATLAB needs nansum() here because log2(0) = -Inf and 0*(-Inf) = NaN.
    scipy.special.xlogy(p, p) returns 0 when p == 0, no warning, no NaN.
    '''
    p = np.asarray(p, dtype=float)
    return float(-np.sum(xlogy(p, p)) / np.log(2))

p = np.array([.5, .5])
print(f"fair coin                  p = {p}   H = {entropy_bits(p):.4f} bits")

p = np.array([.75, .25])
print(f"coin weighted 3:1          p = {p}  H = {entropy_bits(p):.4f} bits")

p = np.full(8, 1/8)
print(f"three flips, 8 outcomes                       H = {entropy_bits(p):.4f} bits")

p = np.full(8, 1/8); p[:4] = 0; p = p / p.sum()
print(f"...told the first flip was heads              H = {entropy_bits(p):.4f} bits")

p = np.full(8, 1/8); p[0] = 0; p = p / p.sum()
red = 3 - entropy_bits(p)
print(f"...told at least one flip was heads           H = {entropy_bits(p):.4f} bits")
print(f"   uncertainty reduction                        = {red:.4f} bits")

A fair coin carries **1 bit** of uncertainty; being told the outcome removes all of it, so
you have learned 1 bit. A coin weighted 3:1 carries only 0.81 bits — learning its outcome
teaches you less. It may seem odd that you learn the same amount from a head as from a tail
when heads are three times as likely; remember that entropy is an *expectation*. A rare
outcome is more surprising, but it happens less often, and the two effects are exactly
balanced in the average.

Entropy is **additive**. Three fair flips have $8$ equally likely outcomes and $3$ bits of
uncertainty. Being told the first flip was heads eliminates half the possibilities and
leaves exactly 2 bits. Being told merely that *at least one* flip came up heads eliminates
only one outcome out of eight and reduces uncertainty by about 0.19 bits — much weaker
information, because it was much more likely to be true.

### Entropy of the interval distributions

Now back to our point processes. For continuous distributions the analogous quantity is the
**differential entropy**

$$h = -\int f(t)\log_2 f(t)\,dt .$$

Two warnings that the original tutorial glosses over, and that matter if you want to use this
number for anything:

1. Differential entropy is **not** the limit of the discrete entropy — it can be negative,
   and it changes if you change units (measuring intervals in seconds rather than
   milliseconds shifts every value below by $\log_2 1000 \approx 9.97$ bits). Only
   *differences* between distributions on the same scale are meaningful, which is precisely
   what we compute.
2. The comparison is only fair if the distributions have the **same mean**. That is why the
   lognormal is re-parameterized to $\mu = 3.412$ below: $e^{3.412 + 0.5} \approx 50$ ms,
   matching the 50 ms mean used for the exponential and gamma cases here. (This is *not* the
   $\mu = 2.5$ lognormal from Example 1, which has a mean of 20 ms.)

In [ ]:
dt = 0.01
t  = np.arange(dt, 1000 + dt/2, dt)     # fine grid, ms

def diff_entropy(pdf_vals, t=t):
    '''Differential entropy in bits by trapezoidal quadrature, after renormalizing.'''
    a = trapezoid(pdf_vals, t)          # should be ~1; slight truncation at the tails
    p = pdf_vals / a
    return trapezoid(-np.log2(np.maximum(p, 1e-300)) * p, t), a, trapezoid(t * p, t)

cases = [
    ("Exponential (Poisson), mu=50",      expon.pdf(t, scale=50)),
    ("Gamma, n=5,  b=10",                 gamma.pdf(t, 5, scale=10)),
    ("Gamma, n=20, b=2.5",                gamma.pdf(t, 20, scale=2.5)),
    ("Lognormal, u=3.412, s=1",           lognorm.pdf(t, 1, scale=np.exp(3.412))),
]
print(f"{'distribution':<30} {'mass on grid':>13} {'mean (ms)':>10} {'CV':>7} {'H (bits)':>10}")
for name, pv in cases:
    h, a, m = diff_entropy(pv)
    p = pv / a
    v = trapezoid((t - m)**2 * p, t)
    print(f"{name:<30} {a:>13.5f} {m:>10.3f} {np.sqrt(v)/m:>7.3f} {h:>10.4f}")

print(f"\nanalytic differential entropy of Exponential(mu=50) = log2(e*mu) = "
      f"{np.log2(np.e*50):.4f} bits")

This is the most interesting result in Part I–II, and it is worth being careful about what it
says.

- The gammas are *less* disorderly than the exponential by entropy (6.43 and 5.51 bits vs
  7.09), which agrees with their CVs.
- The **lognormal is also less disorderly than the exponential** (6.97 vs 7.09 bits) —
  even though its CV of 1.31 said it was *more* disorderly. The CV and the entropy disagree
  outright.

The hazard function told us why back in Example 2: the lognormal has a preferred interval and
becomes more and more predictable the longer you wait. Its heavy tail inflates the CV without
making the process less predictable — in fact it makes it *more* predictable.

The entropy result is not an accident of these parameters. Among all distributions on
$[0,\infty)$ with a given mean $\mu$, the **exponential uniquely maximizes** the differential
entropy, at $\log_2(e\mu)$ bits — which is the number printed at the bottom of the cell, and
which matches the numerical integral. So *any* interval distribution with a 50 ms mean must
come out at or below 7.09 bits. The Poisson process is, in this precise sense, the most
disorderly point process of its rate.

Three measures, three answers: the CV cares about the spread of intervals, the hazard about
conditional predictability, and the entropy about total uncertainty. When someone tells you a
spike train is "noisy", ask which one they mean.

> ### Homework question 3 (optional)
> Write a differential equation that captures the idea that the hazard function equals a
> constant $k$. Let $F$ be the cumulative distribution function, so that $F' = f$ is the
> interval density.
>
> **(a)** Show that $F' = k(1 - F)$, and solve it to show that $f$ must be the exponential
> density.
>
> **(b)** Now do the converse in one line: given $f(t) = \lambda e^{-\lambda t}$, compute
> $f/(1-F)$ directly.
>
> **(c)** Use the same differential equation to construct the interval distribution whose
> hazard is $h(t) = kt$ (a linearly rising hazard). What distribution is it? Where have you
> seen it before?

---
## Part III. Introduction to Markov chains

We now change subject. The processes so far were sequences of stereotyped, interchangeable
events, and we studied the intervals between them. Here we consider processes in which each
event is a **change from one state to another**, and the states have meaning, intensity, or
value. They are not points. By convention we number the states with non-negative integers.

A **Markov chain** has the defining property that the probability of moving from one state to
another does not depend on history. If the sequence reaches state 4, a probability
distribution determines the next state; it does not matter how the process got to state 4.

The bookkeeping convention: the state is a **row** of a matrix, and going across that row,

$$P_{ij} = P(\text{next state is } j \mid \text{current state is } i).$$

$P$ is the **transition matrix**. Its entries lie in $[0,1]$ and every **row sums to 1**.

Consider a three-state ion channel: **open** (1), **closed** (2), **inactivated** (3).

In [ ]:
p_open_from_open   = .7;   p_closed_from_open   = .05;  p_inact_from_open   = .25
p_open_from_closed = .4;   p_closed_from_closed = .6;   p_inact_from_closed = 0
p_open_from_inact  = .1;   p_closed_from_inact  = .3;   p_inact_from_inact  = .6

P = np.array([
    [p_open_from_open,   p_closed_from_open,   p_inact_from_open  ],
    [p_open_from_closed, p_closed_from_closed, p_inact_from_closed],
    [p_open_from_inact,  p_closed_from_inact,  p_inact_from_inact ],
])
states = ["open", "closed", "inactivated"]
print("P =\n", P)

# A proper transition matrix: rows sum to 1, all entries in [0, 1].
ok = np.allclose(P.sum(axis=1), 1) and (P >= 0).all() and (P <= 1).all()
print("\nvalid transition matrix?", ok)
print("row sums:", P.sum(axis=1))

# Start in the CLOSED state -- state 2, which is ROW INDEX 1 in Python.
x0 = np.array([0, 1, 0])
x1 = x0 @ P                       # '@' is matrix multiplication; '*' would be elementwise
print(f"\nstart closed  x0 = {x0}")
print(f"after 1 step  x1 = {x1}    <- this is just row 2 of P: {P[1, :]}")

# Start not knowing: equally likely to be in any state.
x0 = np.ones(3) / 3
x1 = x0 @ P
print(f"\nstart uniform x0 = {np.round(x0, 4)}")
print(f"after 1 step  x1 = {np.round(x1, 4)}   (total probability {x1.sum():.6f})")

Now simulate the actual stochastic process — one channel, one realization — starting from the
closed state. At each step we build the cumulative distribution of the next state and draw a
uniform random number to sample from it. Only the open state passes current.

In [ ]:
nsteps = 100
state_seq = np.zeros(nsteps, dtype=int)      # 0 = open, 1 = closed, 2 = inactivated
s = np.array([0, 1, 0])                      # initial state: closed

for n in range(nsteps):
    p_s = np.cumsum(s @ P)                   # cumulative distribution of the next state
    xr  = rng.random()                       # uniform on [0, 1); sample from p_s
    k   = int(np.searchsorted(p_s, xr))      # 0, 1 or 2  (the if/elseif ladder, vectorized)
    s   = np.eye(3, dtype=int)[k]
    state_seq[n] = k

y = (state_seq == 0).astype(int)             # 1 while the channel passes current

fig, axes = plt.subplots(2, 1, figsize=(10, 4.2), sharex=True,
                         gridspec_kw={"height_ratios": [1.4, 1]})
axes[0].step(np.arange(nsteps), state_seq, where="post", color=PURPLE, lw=1.2)
axes[0].set(yticks=[0, 1, 2], ylim=(-0.4, 2.4), ylabel="State")
axes[0].set_yticklabels(states)
axes[0].set_title("One simulated channel: state sequence")
axes[1].step(np.arange(nsteps), y, where="post", color=BLUE, lw=1.2)
axes[1].fill_between(np.arange(nsteps), 0, y, step="post", color=BLUE, alpha=0.3)
axes[1].set(yticks=[0, 1], ylim=(-0.1, 1.3), xlabel="Time step", ylabel="Current")
axes[1].set_yticklabels(["shut", "open"])
axes[1].set_title("Current passed (only the open state conducts)")
fig.tight_layout()

for k, nm in enumerate(states):
    print(f"fraction of steps in the {nm:<12s} state: {(state_seq == k).mean():.3f}")

Run that cell again and the picture changes: it is a simulation, and the only thing that
enters is the stream of uniform random numbers `rng.random()`.

> ### Homework question 4
> Where, exactly, does the randomness enter the simulation above?
>
> **(a)** The transition matrix `P` contains no random numbers at all, yet the output differs
> every time. Identify the single line responsible, and explain what `np.searchsorted` on the
> cumulative distribution is doing.
>
> **(b)** The state vector `s` is always a one-hot vector like `[0 1 0]` in the simulation,
> but `x0 @ P` above used a vector of *probabilities* like `[1/3 1/3 1/3]`. What is the
> difference in interpretation? Which one describes a single channel and which describes a
> population of channels?
>
> **(c)** The fraction of steps spent open, printed above, is an estimate of something. Of
> what? How would you put an error bar on it, given that successive samples are *not*
> independent?

### The distribution after $n$ steps

Instead of one realization, track the whole probability distribution. After one step
$s_1 = s_0 P$, after two $s_2 = s_1 P = (s_0 P)P = s_0 P^2$, and in general $s_n = s_0 P^n$.

**This is the place to be careful in Python.** MATLAB's `P^n` is the matrix power; NumPy's
`P**n` raises **every element** to the $n$-th power and reports no error. Use
`np.linalg.matrix_power(P, n)` (or repeated `@`).

In [ ]:
s0 = np.array([1.4, 1.2, 0.4]) / 3
print("s0 =", np.round(s0, 4), " (sums to", s0.sum(), ")")
for n in (1, 2, 3):
    print(f"s{n} = {np.round(s0 @ np.linalg.matrix_power(P, n), 5)}")

print("\nthe elementwise trap:")
print("np.linalg.matrix_power(P, 3) =\n", np.round(np.linalg.matrix_power(P, 3), 5))
print("P ** 3  (ELEMENTWISE - wrong here) =\n", np.round(P**3, 5))
print("\nrow sums of the elementwise version:", np.round((P**3).sum(axis=1), 4),
      " <- not 1, so it is not a transition matrix at all")
print("MATLAB's P.^3 is the elementwise one; MATLAB's P^3 is the matrix one. "
      "NumPy reverses this.")

In [ ]:
nmax = 50
S = np.array([s0 @ np.linalg.matrix_power(P, i) for i in range(1, nmax + 1)])

fig, ax = plt.subplots(figsize=(8, 3.8))
for k, (nm, col) in enumerate(zip(states, (BLUE, PURPLE, GREEN))):
    ax.plot(np.arange(1, nmax + 1), S[:, k], color=col, lw=1.8, label=nm)
# same chain from three quite different starting distributions, open-state probability only
for s_alt, ls in ((np.array([1., 0, 0]), "--"), (np.ones(3)/3, ":"),
                  (np.array([0., 0, 1]), "-.")):
    traj = [s_alt @ np.linalg.matrix_power(P, i) for i in range(1, nmax + 1)]
    ax.plot(np.arange(1, nmax + 1), [v[0] for v in traj], ls, color=BLUE, lw=1, alpha=0.7)
ax.set(xlabel="Time step", ylabel="Probability of being in state",
       ylim=(0, 0.75), xlim=(1, nmax),
       title="Convergence to steady state (thin blue = P(open) from other starting states)")
ax.legend(loc="upper right")
fig.tight_layout()

print("distribution after 50 steps:", np.round(S[-1], 6))
print(f"steady-state P(open) = {S[-1, 0]:.6f}")
print(f"fraction of steps open in the single-channel simulation above: {y.mean():.3f}")

Start anywhere you like and you land in the same place. The dynamics are sensible if you
think of the probabilities as the *fraction of a large population of channels* in each state:
starting with everything closed, the channels surge through a transient in which well over
half are open before settling down.

**A correction to the original.** The MATLAB comment says the steady state is "about 42% of
channels in the open state"; a later comment in the same file says 0.45. The second is right
— the exact steady-state open probability is 32/71 = 0.450704…, which we confirm both by
brute-force iteration above and by the eigenvector calculation below. The single-channel
simulation's fraction of open steps is a noisy estimate of the same quantity (100 correlated
samples, so its error bar is wider than $\sqrt{pq/n}$ would suggest).

### The eigensystem of the transition matrix

We can see *why* the chain converges — and to what — from the eigenvectors of the transition
matrix. It helps to transpose first. So far, rows have been the "from" states and columns the
"to" states, so the update is a *row* vector times $P$ on the right. Define

$$T = P^{\mathsf{T}}$$

so that columns are "from" and rows are "to". Now the update is an ordinary matrix–vector
product on a **column** vector, $s_{n} = T\,s_{n-1}$, and hence $s_n = T^n s_0$. The
**columns** of $T$ sum to 1.

In [ ]:
T = P.T
print("T = P' =\n", T)
print("column sums of T:", T.sum(axis=0))

s0c = s0.reshape(-1, 1)                       # column vector
print("\nT @ s0  =", np.round((T @ s0c).ravel(), 5))
print("s0 @ P  =", np.round(s0 @ P, 5), "  <- identical, as it must be")
print("T^10 @ s0 =", np.round((np.linalg.matrix_power(T, 10) @ s0c).ravel(), 6))

eigVals, eigVecs = np.linalg.eig(T)           # MATLAB: [eigVecs eigVals] = eig(T)
print("\neigenvalues:", np.round(eigVals, 5))
for i in range(3):
    resid = np.abs(T @ eigVecs[:, i] - eigVals[i] * eigVecs[:, i]).max()
    print(f"  |T v{i} - lambda{i} v{i}|_max = {resid:.2e}")

Do not be put off by the complex numbers: two of the eigenvalues here are a complex-conjugate
pair, $0.45 \pm 0.229i$, with modulus $|\lambda| \approx 0.505$. Complex eigenvalues of a
transition matrix mean the approach to steady state has an **oscillatory** component, which
is exactly the overshoot visible in the convergence figure.

One eigenvalue is exactly 1 — every transition matrix has one, because the rows of $P$ sum to
1 — and, conveniently, its eigenvector is real. Scale it so its elements sum to 1 and it is a
legitimate probability distribution, unchanged by the transition: the **steady state**.

In [ ]:
i1   = int(np.argmin(np.abs(eigVals - 1)))
eig1 = np.real(eigVecs[:, i1])
eig1 = eig1 / eig1.sum()

print("eigenvector with eigenvalue 1, normalized to sum to 1:")
for nm, v in zip(states, eig1):
    print(f"   P({nm:<12s}) = {v:.6f}")
print(f"\n   sums to {eig1.sum():.6f}")
print("   T @ eig1 =", np.round(T @ eig1, 6), " <- returns itself, so it is a fixed point")
print(f"\n   brute-force iteration after 50 steps: {np.round(S[-1], 6)}")
print(f"   max |eigenvector - iterated| = {np.abs(eig1 - S[-1]).max():.2e}")
print(f"   exact value for P(open): 32/71 = {32/71:.6f}")

So far `eig1` is only a *candidate*: we know that if we start there we stay there. To see that
we are guaranteed to **get** there, look at the matrix power through the eigendecomposition.

Any diagonalizable $T$ factors as $T = V\Lambda V^{-1}$, where $V$ holds the eigenvectors as
columns and $\Lambda$ is diagonal. Then

$$T^2 = V\Lambda V^{-1}V\Lambda V^{-1} = V\Lambda^2 V^{-1},
\qquad\text{and in general}\qquad T^n = V\Lambda^n V^{-1}.$$

Since $\Lambda$ is diagonal, raising it to the $n$-th power just raises the eigenvalues.
Every eigenvalue with $|\lambda| < 1$ therefore **decays away**, the faster the smaller it is,
leaving only the $\lambda = 1$ eigenvector — no matter what $s_0$ was.

In [ ]:
V, Lam = eigVecs, np.diag(eigVals)
recon  = V @ Lam @ np.linalg.inv(V)
print("V Lam inv(V) reproduces T:      max |error| =", f"{np.abs(recon - T).max():.2e}")
print("  (imaginary parts are numerical dust: max |Im| =",
      f"{np.abs(np.imag(recon)).max():.2e})")

# Three ways to write the square. Note Lam**2 IS correct here only because Lam is DIAGONAL.
print("\nT @ T                              =\n", np.round(T @ T, 5))
print("np.linalg.matrix_power(T, 2)       =\n", np.round(np.linalg.matrix_power(T, 2), 5))
print("real(V @ Lam**2 @ inv(V))          =\n",
      np.round(np.real(V @ Lam**2 @ np.linalg.inv(V)), 5))

print("\ndecay of each eigen-mode, |lambda|^n:")
print(f"{'n':>4}" + "".join(f"{f'|l{i}|^n':>12}" for i in range(3)))
for n in (1, 5, 10, 20, 40):
    print(f"{n:>4}" + "".join(f"{abs(eigVals[i])**n:>12.2e}" for i in range(3)))

The $\lambda = 1$ mode is immortal; the other two are down by a factor of $10^{-12}$ after 40
steps. That is the whole story of convergence, and it also tells you the **rate**: the
approach to steady state is governed by the second-largest modulus, here $0.505$, giving a
time constant of $-1/\ln(0.505) \approx 1.5$ steps.

> **Note.** `Lam**2` above is correct *only* because `Lam` is diagonal, where elementwise and
> matrix powers coincide. Written for a general matrix, `V @ Lam**2 @ inv(V)` would be
> quietly wrong. When in doubt, use `np.linalg.matrix_power`.

### Digression: what else matrix powers are good for

Factoring a matrix into its eigensystem generalizes a great deal of scalar math to higher
dimensions. Matrices have square roots and exponentials, and both are computed by doing the
scalar operation on the eigenvalues.

In [ ]:
A = np.array([[1., 2.], [3., 4.]])
Q = scipy.linalg.sqrtm(A)                 # MATLAB sqrtm
lamA, VA = np.linalg.eig(A)
print("sqrtm(A) =\n", np.round(Q, 5))
print("Q @ Q    =\n", np.round(np.real(Q @ Q), 5), "  <- back to A")
print("V sqrt(Lam) inv(V) =\n",
      np.round(np.real(VA @ np.diag(np.sqrt(lamA.astype(complex))) @ np.linalg.inv(VA)), 5))

print("\nexpm(T) =\n", np.round(scipy.linalg.expm(T), 5))
print("V diag(exp(lam)) inv(V) =\n",
      np.round(np.real(V @ np.diag(np.exp(eigVals)) @ np.linalg.inv(V)), 5))
print("\n(np.exp(T), the ELEMENTWISE exponential, is something else entirely:\n",
      np.round(np.exp(T), 5), ")")

One more, for fun. Consider

$$A = \begin{pmatrix} 1 & 1 \\ 1 & 0\end{pmatrix}, \qquad u_0 = \begin{pmatrix}1\\0\end{pmatrix}.$$

Applying $A$ repeatedly generates the **Fibonacci** sequence: $u_k$ holds the $(k{+}1)$-th and
$k$-th Fibonacci numbers, and each new number is the sum of the previous two. Because
$u_n = A^n u_0$, and $A^n = V\Lambda^n V^{-1}$, you can jump straight to the $n$-th Fibonacci
number without iterating. The larger eigenvalue is the golden ratio.

In [ ]:
A  = np.array([[1., 1.], [1., 0.]])
u0 = np.array([1., 0.])
lamF, VF = np.linalg.eig(A)
print("eigenvalues of A:", np.round(lamF, 6), "   golden ratio =", f"{(1+np.sqrt(5))/2:.6f}")

u = u0.copy()
seq = [u[1]]
for _ in range(9):
    u = A @ u
    seq.append(u[1])
print("iterating A on u0:", [int(round(v)) for v in seq])

for n in (5, 6, 12):
    direct = np.real(VF @ np.diag(lamF**n) @ np.linalg.inv(VF) @ u0)
    print(f"V Lam^{n:>2} inv(V) u0 = {np.round(direct, 4)}   "
          f"-> Fibonacci #{n+1} and #{n} = {int(round(direct[0]))}, {int(round(direct[1]))}")

---
### A random walk with absorbing barriers: gambler's ruin

Finally, a Markov chain that approximates **diffusion**. Fred and Adrienne each have \$4.
They bet a dollar and flip a coin: heads, Fred wins a dollar from Adrienne; tails, the
reverse. The game ends when one player has \$8 and the other has nothing.

Define 9 states by relative winnings: Fred ahead by 4, by 3, …, even, …, Adrienne ahead by 4.
The transition matrix has $p$ on the superdiagonal and $q = 1-p$ on the subdiagonal, except in
the first and last rows, which have a 1 on the main diagonal: the probability of *leaving*
those states is zero. They are **absorbing states** — the game is over. From any intermediate
state there is no chance of staying put.

This is the discrete analogue of a particle diffusing toward absorbing barriers placed
symmetrically about the origin.

*(A note on the original: the MATLAB text says "let's begin by assuming that the coin is
fair" and then sets `p = .6`, which is not fair. We do the fair case first, as the text
intends, and the biased case after.)*

In [ ]:
def ruin_matrix(p, nstates=9):
    '''Transition matrix for a random walk with absorbing barriers at both ends.'''
    q = 1 - p
    P = np.diag(np.full(nstates - 1, p), 1) + np.diag(np.full(nstates - 1, q), -1)
    P[0, :]  = 0; P[0, 0]   = 1          # absorbing
    P[-1, :] = 0; P[-1, -1] = 1          # absorbing
    return P

P_fair = ruin_matrix(0.5)
print("fair game, P =\n", P_fair)
print("\nrow sums:", P_fair.sum(axis=1), "  all equal 1?", np.allclose(P_fair.sum(axis=1), 1))
print("\nT = P' (columns are the 'from' states):\n", P_fair.T)

In [ ]:
labels = ["Fred +4", "Fred +3", "Fred +2", "Fred +1", "even",
          "Adr +1", "Adr +2", "Adr +3", "Adr +4"]
start = np.zeros(9); start[4] = 1               # state 5 == index 4: both have $4

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, pval in zip(axes, (0.5, 0.6)):
    Pm = ruin_matrix(pval)
    d16 = start @ np.linalg.matrix_power(Pm, 16)
    ax.bar(np.arange(9), d16, color=[RED if i in (0, 8) else BLUE for i in range(9)],
           edgecolor="white")
    ax.set_xticks(np.arange(9)); ax.set_xticklabels(labels, rotation=60, ha="right")
    ax.set(ylim=(0, 0.45), title=f"After 16 rounds, p = {pval}")
    ax.set_ylabel("Probability" if pval == 0.5 else "")
    print(f"p = {pval}:  distribution after 16 rounds = {np.round(d16, 5)}")
    print(f"          P(game over)   = {d16[0] + d16[8]:.5f}"
          f"   [Fred wins {d16[0]:.5f}, Adrienne wins {d16[8]:.5f}]")
    print(f"          states with probability exactly 0: "
          f"{[labels[i] for i in np.flatnonzero(np.isclose(d16, 0))]}\n")
fig.suptitle("Gambler's ruin: red bars are the absorbing states", y=1.04)
fig.tight_layout()

> ### Homework question 5
> **(a)** What is the probability distribution of states after 16 rounds of play if the game
> begins in state 5 (both players have \$4) and the coin is fair? The cell above computes it —
> your job is to explain it.
>
> **(b)** What is the probability that the game has ended by 16 rounds?
>
> **(c)** The probability is exactly 0 for four of the nine states. Which four, and *why*?
> (Think about what one round does to the parity of the state index, and about why the
> absorbing states are exempt from the argument.)
>
> **(d)** Now suppose the coin unfairly favours Adrienne, $p = 0.6$. What is the probability
> that the game has ended by round 16? What is the probability that it has ended with
> Adrienne the winner? Compare with the fair case and explain the direction of the change.

Now look at the eigensystem of this chain. Unlike the ion channel, it has **two** eigenvalues
equal to 1 — one for each absorbing state. There is no unique steady state; where you end up
depends on where you started.

In [ ]:
for pval in (0.5, 0.6):
    T = ruin_matrix(pval).T
    ev, EV = np.linalg.eig(T)
    order = np.argsort(-np.abs(ev))
    ev, EV = ev[order], EV[:, order]
    print(f"p = {pval}: eigenvalue moduli = {np.round(np.abs(ev), 4)}")
    print(f"          number of eigenvalues equal to 1: {np.isclose(ev, 1).sum()}")
    for j in np.flatnonzero(np.isclose(ev, 1)):
        v = np.real(EV[:, j]); v = v / v.sum()
        print(f"          equilibrium eigenvector: {np.round(v, 4)}"
              f"  -> {labels[int(np.argmax(np.abs(v)))]} absorbed")
    print()

# Long-run absorption probabilities, by brute force and in closed form.
print("absorption probabilities starting from 'even' (state 5):")
for pval in (0.5, 0.6):
    d = start @ np.linalg.matrix_power(ruin_matrix(pval), 4000)
    if np.isclose(pval, 0.5):
        exact = 0.5
    else:
        r = (1 - pval) / pval          # ratio for the classic gambler's-ruin formula
        exact = (1 - r**4) / (1 - r**8)
    print(f"   p = {pval}:  P(Fred wins) = {d[0]:.6f}   (closed form {exact:.6f}),"
          f"   P(Adrienne wins) = {d[8]:.6f}")

The two unit eigenvalues correspond to the two absorbing states, and any starting
distribution is eventually split between them. The classic gambler's-ruin formula for a
walker starting $a$ steps from ruin with $N$ total steps between barriers,

$$P(\text{reach } 0 \text{ first}) = \frac{1 - r^{a}}{1 - r^{N}}, \qquad r = q/p,$$

matches the brute-force matrix powers exactly. A modest bias — 0.6 instead of 0.5 — turns a
coin flip into near-certainty over eight steps of separation. This is the same mathematics
that governs the drift-diffusion model of decision making: bounded accumulation of noisy
evidence, where the drift rate plays the role of the bias and the bound separation plays the
role of the stakes.

> ### Homework question 6 (extra credit)
> What is the expected number of steps for the **fair** game to end? What is the standard
> deviation of that number?
>
> **(a)** Derive the probabilities empirically — simulate many games and take the first and
> second moments of the duration.
>
> **(b)** Get the same answer from the transition matrix without simulating, by tracking how
> much probability mass is absorbed at each step: $P(\text{ends at step } n)$ is the increment
> in $d_1 + d_9$ from step $n-1$ to step $n$.
>
> **(c)** For the fair game with $\pm a$ barriers there is a famous closed form for the mean
> duration, $\mathbb{E}[N] = a^2$ when the walk starts in the middle. Does your answer agree?
> What happens to the mean duration when $p = 0.6$ — longer or shorter? Why?

---
## Summary

1. A **point process** is a list of indistinguishable events. A **renewal process** is one
   whose intervals are iid — every event restarts the process.

2. Three descriptions of the same process: the **interval distribution**, the **count
   distribution**, and the **hazard function**. For a renewal process they are linked
   asymptotically: $\mathbb{E}[N] \to T/\mu_I$, $\mathrm{Var}[N] \to T\sigma_I^2/\mu_I^3$, and
   therefore Fano factor $\to \mathrm{CV}^2$. The mean relation converges fast, the variance
   relation slowly.

3. The **Poisson process** — exponential intervals — is the benchmark. Its CV is 1, its Fano
   factor is 1, its hazard is flat at $1/\mu$, and it is **memoryless**. Every one of those
   is a statement about the distribution; a finite sample will always miss by a percent or
   several, and the simulations above show exactly how much.

4. **Gamma** intervals arise mechanistically from integrating $n$ Poisson EPSPs to threshold,
   with $\mathrm{CV} = 1/\sqrt{n}$ and a hazard that starts at zero and saturates at the EPSP
   rate. The model's problem is that it is *too regular* to match cortex.

5. **Disorderliness is not one thing.** For our lognormal process, the CV said "more
   disorderly than Poisson" while the hazard and the entropy both said "less". The
   exponential maximizes differential entropy among all interval distributions of a given
   mean, so no renewal process of a given rate can be more disorderly by that measure.

6. A **Markov chain** moves between meaningful states with history-independent probabilities.
   Iterating it is repeated multiplication by the transition matrix; the eigensystem explains
   both the destination (the $\lambda=1$ eigenvector is the steady state) and the approach
   (modes decay as $|\lambda|^n$). With **absorbing states** there are several unit
   eigenvalues and no unique steady state — as in gambler's ruin, the discrete cousin of
   bounded diffusion.

7. The porting trap worth repeating: **`A**n` in NumPy is elementwise.** Use
   `np.linalg.matrix_power`. It never errors; it just gives you a wrong answer that looks
   like a matrix.

### Further reading

- Berg, H. C. *Random Walks in Biology* — the source for the probability review this tutorial
  assumes, and for the diffusion picture behind Part III.
- Cox, D. R. (1962). *Renewal Theory.* Methuen. The count/interval asymptotics of Part I.
- Rieke, Warland, de Ruyter van Steveninck & Bialek (1997). *Spikes: Exploring the Neural
  Code.* MIT Press. Entropy and information in spike trains, done properly.
- Softky & Koch (1993). The highly irregular firing of cortical cells is inconsistent with
  temporal integration of random EPSPs. *J. Neurosci.* **13**, 334–350. The Example 3
  problem, stated as a problem.
- Shadlen & Newsome (1998). The variable discharge of cortical neurons: implications for
  connectivity, computation, and information coding. *J. Neurosci.* **18**, 3870–3896. One
  resolution: balanced excitation and inhibition.
- Gardiner, C. W. *Handbook of Stochastic Methods* — Markov chains, master equations, and
  first-passage problems in one place.